In [ ]:
import torch
from typing import Set, Type
from trainer.trainer import RFDiffusionTrainer
from configs.config import DiffusionConfig
from architectures.build_architecture import build_architecture
from dataloaders.build_dataloader import build_dataset, build_dataloader
from augmentations.vision_augmentation import get_image_processor
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from utils.utils import combine_tensors

In [ ]:
config = DiffusionConfig()

In [ ]:
model = build_architecture(config.model)
diffusion_model = model["model"].to("cuda:0")
vae = model["vae"].to("cuda:0")
time_sampler = model["time_sampler"]
caption_encoder = model["caption_encoder"].to("cuda:0")

In [ ]:
diffusion_model.load_state_dict(
    torch.load(
        "PATH",
        weights_only=False,
    )
)

In [ ]:
val_dataset = build_dataset(
    config, False, get_image_processor(config.model.vae_input_size)
)
val_dataloader = build_dataloader(config.training, val_dataset)

In [ ]:
import torch
import torch.nn.functional as F
from tqdm import tqdm


def generate_rectified_flow(
    model,
    shape,
    c,
    c_pooled,
    steps=100,
    t_min=0.0,
    t_max=1.0,
    device="cuda",
    sampling_method="euler",
):
    """
    Generate a sample from a rectified flow model by integrating the velocity field.

    Args:
        model: The rectified flow model (velocity predictor).
        shape: The shape of the latent variable to generate (e.g., [batch, channels, height, width]).
        cond: Conditioning input (text embeddings, image features, etc).
        steps: Number of integration steps (higher = better quality).
        t_min: Start time (typically 0.0).
        t_max: End time (typically 1.0).
        device: Torch device.
        sampling_method: Integration method: "euler" or "rk4" (Runge-Kutta).
    Returns:
        A generated latent sample.
    """
    x = torch.randn(shape, device=device)  # Start from pure noise
    time_steps = torch.linspace(t_max, t_min, steps + 1, device=device)

    for i in tqdm(range(steps), desc="Sampling with rectified flow"):
        t1 = time_steps[i]
        t2 = time_steps[i + 1]
        dt = t2 - t1

        t1_batch = t1.expand(x.shape[0])  # if time is expected as a batch tensor

        if sampling_method == "euler":
            with torch.no_grad():
                v, _ = model(x, c, c_pooled, t1_batch)
            x = x + v * dt
        elif sampling_method == "rk4":
            # Runge-Kutta 4th order method
            def velocity(x_, t_):
                return model(x_, t_.expand(x_.shape[0]), cond=None)

            k1 = velocity(x, t1)
            k2 = velocity(x + 0.5 * dt * k1, t1 + 0.5 * dt)
            k3 = velocity(x + 0.5 * dt * k2, t1 + 0.5 * dt)
            k4 = velocity(x + dt * k3, t1 + dt)

            x = x + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

        else:
            raise ValueError(f"Unknown sampling method: {sampling_method}")

    return x

In [ ]:
for batch in val_dataloader:
    images = batch["image_data"]
    text = batch["text_data"]

    # sample time
    t = time_sampler(images.shape[0])

    # encode text
    encoded_text = caption_encoder(text, device="cuda:0")
    c, c_pooled = combine_tensors(encoded_text)
    break

In [ ]:
index = 8

out = generate_rectified_flow(
    diffusion_model.to("cuda:0").eval(),
    [1, 16, 32, 32],
    c[index].to("cuda:0"),
    c_pooled[index].to("cuda:0"),
    steps=100,
    device="cuda:0",
)

In [ ]:
print(text[index])

In [ ]:
scaled_output = (
    out.to(vae.vae.dtype) - vae.vae.config.shift_factor
) / vae.vae.config.scaling_factor


vae.eval()
# Decode latent to image
with torch.no_grad():
    image = vae.decode(scaled_output.detach())[0]  # scale if using Stable Diffusion VAE

# image = torch.clamp(image, -1, 1)
# image = image * 0.5 + 0.5

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 15))
axes[0].imshow(images[index].moveaxis(0, 2).numpy())
axes[1].imshow(image.moveaxis(0, 2).float().detach().cpu().numpy())